# Q-factorisation on Gridworld Maze- Baseline, full retrain not regularisation

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from copy import deepcopy


# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.maze_discrete import MazeGridWorld, MazeGoalWrapper
from utils import (TrajectoryReplayBufferDiscrete, evaluate_policy, set_seed, build_goal_batch, get_base_env, collect_valid_states_fourrooms,
                   estimate_fisher_diag, extract_fixed_probe_sa_embedding, extract_mean_sa_embedding, extract_sa_batch_for_isotropy,
                   compute_embedding_drift, collect_weight_snapshot)
from visualisations import visualise_embeddings, visualise_q_table, print_goal_embedding_similarity, plot_full_embedding_dashboard_html
from loss_functions import repulsion_loss_to_memory, sigreg_loss, orthogonal_loss, ewc_regulariser_loss, weight_regulariser_loss
from networks import snapshot_named_parameters, FactorisedDQN_QNetwork
from trainer import dqn_train


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
MAZE_LAYOUT = [
    [1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,1],
    [1,1,1,0,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1],
]

def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=500):
    base = MazeGridWorld(
        maze=MAZE_LAYOUT,
        max_episode_steps=max_horizon,
    )
    env = MazeGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
        reward_mode="simple",
    )
    return env

env = make_env(goal=(9, 9))
obs, info = env.reset()

img = env.unwrapped.render()
goal = env.goal_position

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(
    goal[0] * 40 + 20,
    goal[1] * 40 + 20,
    c="lime",
    s=180,
    marker="*",
    edgecolors="black",
)
plt.title(f"Maze with goal at {goal}")
plt.axis("off")
plt.show()

## Training Loop across goals

first loop = retrain from scratch for every goal

second loop = retrain from previous task weights as initialisation

In [ ]:
SEEDS = [42]
PASS = 2
GOALS = [(9, 9), (7, 7), (3, 5), (3, 9), (5, 5), (3,6), (9,1), (1,1), (7,4), (9,8),(9,7)]
BUFFER_CAPACITY = 100000
LR = float(1e-3)
sa_keywords_local = ["sa_encoder"]
goal_keywords_local = ["goal_encoder"]

overall_results = {
    goal: {
        "eval_returns": [],
        "eval_returns_time": [],
        "min_steps": [],
        "min_time": [],
        "task_embeddings": [],
        "sa_embeddings": [],
        "sa_fixed_probe_embeddings": [],
        "sa_batches_final": [],
    }
    for goal in GOALS
}

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================\n")
    set_seed(seed)  # your helper

    # Create env to get obs_dim, num_actions once
    env_tmp = make_env(goal=GOALS[0])
    obs_dim = env_tmp.observation_space.shape[0]
    num_actions = env_tmp.action_space.n
    env_tmp.close()

    # One network per seed, reused across goals
    q_net_base = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)

    q_target_base = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)


    # Per-seed task embedding memory (for repulsion)
    seed_task_embedding_memory = []
    seen_goal_labels = []
    weight_history = []
    prev_q_net = None
    prev_q_target = None 

    for pass_num in range(PASS):
        print(f"\n===== PASS {pass_num + 1} / {PASS} =====\n")
        for goal_idx, goal in enumerate(GOALS):
            print(f"\n----- seed={seed}, goal={goal} -----\n")

            if pass_num == 0 or goal_idx == 0:
                q_net = deepcopy(q_net_base)
                q_target = deepcopy(q_target_base)
                q_target.load_state_dict(q_net.state_dict())
                for p in q_target.parameters():
                    p.requires_grad_(False)
            else:
                if prev_q_net is None or prev_q_target is None:
                    raise ValueError("Previous Q-networks are not available for pass > 0.")
                q_net = deepcopy(prev_q_net)
                q_target = deepcopy(prev_q_target)
                q_target.load_state_dict(q_net.state_dict())
                for p in q_target.parameters():
                    p.requires_grad_(False)

            weight_history.append(
                collect_weight_snapshot(
                    qnet=q_net,
                    goal_label=goal,
                    stage_label=f"goal_{goal_idx}_before_train_{goal}",
                    sa_keywords_local=sa_keywords_local,
                    goal_keywords_local=goal_keywords_local,
                    max_samples_per_group=40000,
                )
            )

            (
                q_network,
                q_target_network,
                eval_returns,
                eval_returns_time,
                min_steps,
                min_time,
                task_embedding,
                sa_embedding_mean,
                sa_embedding_fixed,
                sa_batch_final,
                buffer,
            )= dqn_train(
                q_network=q_net,
                q_target_network=q_target,
                env=make_env(goal=goal),
                obs_dim=obs_dim,
                buffer_capacity=BUFFER_CAPACITY,
                lr=LR,
                goal=goal,
                device=DEVICE,
                embedding_memory=seed_task_embedding_memory,  # previous task embeddings in this seed
                regulariser=None,
                reg_alpha=1,          # your choice 
                make_env=make_env
            )

            weight_history.append(
                collect_weight_snapshot(
                    qnet=q_network,
                    goal_label=goal,
                    stage_label=f"goal_{goal_idx}_after_train_{goal}",
                    sa_keywords_local=sa_keywords_local,
                    goal_keywords_local=goal_keywords_local,
                    max_samples_per_group=40000,
                )
            )
            visualise_q_table(goal, q_network, eval_returns=eval_returns, device=DEVICE, make_env=make_env)
            visualise_embeddings(goal, q_network, device=DEVICE, make_env=make_env)
            seed_task_embedding_memory.append(task_embedding)
            seen_goal_labels.append(str(goal))
            print_goal_embedding_similarity(seed_task_embedding_memory, goal_labels=seen_goal_labels)

            if pass_num != 0:
                prev_q_net = deepcopy(q_network)
                prev_q_target = deepcopy(q_target_network)


            overall_results[goal]["eval_returns"].append(eval_returns)
            overall_results[goal]["eval_returns_time"].append(eval_returns_time)
            overall_results[goal]["min_steps"].append(min_steps)
            overall_results[goal]["min_time"].append(min_time)
            overall_results[goal]["task_embeddings"].append(task_embedding)
            overall_results[goal]["sa_embeddings"].append(sa_embedding_mean)
            overall_results[goal]["sa_fixed_probe_embeddings"].append(sa_embedding_fixed)
            overall_results[goal]["sa_batches_final"].append(sa_batch_final)


In [ ]:
# --------------------------------------------------
# CHANGE THESE NAMES ONLY
# Must match the number of experiments per goal
# Example: ["seed_1", "seed_2"] or ["exp_A", "exp_B", "exp_C"]
experiment_labels = ["retrain_from_scratch", "retrain_from_previous_tasks_initialisation"]
# --------------------------------------------------

goals = list(overall_results.keys())

used_goals = []
steps_per_goal = []
time_per_goal = []

for goal in goals:
    steps_raw = overall_results[goal]["min_steps"]
    time_raw  = overall_results[goal]["min_time"]

    steps_arr = np.asarray(steps_raw).reshape(-1)
    time_arr  = np.asarray(time_raw).reshape(-1)

    # Skip goals with no data
    if steps_arr.size == 0 or time_arr.size == 0:
        print(f"[WARN] Skipping goal {goal}: empty min_steps or min_time")
        continue

    # Require same number of experiments for steps/time for this goal
    if steps_arr.size != time_arr.size:
        print(
            f"[WARN] Skipping goal {goal}: "
            f"min_steps has {steps_arr.size} entries but min_time has {time_arr.size}"
        )
        continue

    steps_per_goal.append(steps_arr)
    time_per_goal.append(time_arr)
    used_goals.append(goal)

if not used_goals:
    raise ValueError("No goals with valid non-empty min_steps/min_time found.")

# Convert to arrays of shape: (num_goals, num_experiments)
steps_per_goal = np.vstack(steps_per_goal)
time_per_goal  = np.vstack(time_per_goal)

num_goals, num_experiments = steps_per_goal.shape
x = np.arange(num_goals)

if len(experiment_labels) != num_experiments:
    raise ValueError(
        f"experiment_labels has length {len(experiment_labels)}, "
        f"but data contains {num_experiments} experiments per goal."
    )

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.set_title("Minimum steps and time to reach a good return (per goal)")

# Left y-axis: min steps
step_colors = plt.cm.Blues(np.linspace(0.45, 0.9, num_experiments))
step_lines = []

for exp_idx in range(num_experiments):
    line, = ax1.plot(
        x,
        steps_per_goal[:, exp_idx],
        label=f"Steps - {experiment_labels[exp_idx]}",
        color=step_colors[exp_idx],
        marker="o",
        linestyle="-",
    )
    step_lines.append(line)

ax1.set_xlabel("Goal Index")
ax1.set_ylabel("Min Steps", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Right y-axis: min time
ax2 = ax1.twinx()
time_colors = plt.cm.Oranges(np.linspace(0.45, 0.9, num_experiments))
time_lines = []

for exp_idx in range(num_experiments):
    line, = ax2.plot(
        x,
        time_per_goal[:, exp_idx],
        label=f"Time - {experiment_labels[exp_idx]}",
        color=time_colors[exp_idx],
        marker="s",
        linestyle="--",
    )
    time_lines.append(line)

ax2.set_ylabel("Min Time", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

# Label x-axis with actual goal IDs
ax1.set_xticks(x)
ax1.set_xticklabels([str(g) for g in used_goals], rotation=45, ha="right")

# Combined legend across both axes
lines = step_lines + time_lines
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="best")

ax1.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## HTML visualisation

In [ ]:
for goal, data in overall_results.items():
    print(goal)
    print("task_embeddings:", len(data.get("task_embeddings", [])))
    print("sa_fixed_probe_embeddings:", len(data.get("sa_fixed_probe_embeddings", [])))
    print("sa_batches_final:", len(data.get("sa_batches_final", [])))

dashboard = plot_full_embedding_dashboard_html(
    overall_results=overall_results,
    qnet=q_net,
    mode="all",
    save_html="plots/full_embedding_dashboard_maze_baseline.html",
    weights_his=weight_history
)

print(dashboard["save_html"])
print(dashboard["isotropy_metrics"])